In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import os, warnings 
warnings.filterwarnings('ignore') 

from setup import *
from glob import glob
import stan

## Loading the dataset

In [2]:
sample_list = []
for x in glob('data/Ren_processed/*'):
    sample_list.append(x.split(r'/')[-1][:-5])
sample_list.sort()
sample_list

['Ren2', 'Ren3', 'Ren51']

In [3]:
adatas_tfa = dict()
adatas_pwa = dict() 

for sample in sample_list:
    adata = sc.read_h5ad(f"results/ren_tfa/{sample}.h5ad")
    adatas_tfa[sample] = stan.get_activity(adata, key='tfa_stan')
    
    adata = sc.read_h5ad(f"results/ren_pwa/{sample}.h5ad")
    adatas_pwa[sample] = stan.get_activity(adata, key='pwa_stan')

In [4]:
df_lr_pair = pd.read_csv('resources/human_lr_pair.csv', index_col=0)
lr_genes = list(
    set(df_lr_pair['ligand_gene_symbol']) | 
    set(df_lr_pair['receptor_gene_symbol']))

In [5]:
tf_list = pd.Index([])
for sample in sample_list:
    tf_list = tf_list.union(adatas_tfa[sample].var_names)
tf_list = tf_list.to_list()

In [6]:
adatas_tfmrna = dict()
adatas_markers = dict()
adatas_lrs = dict()
for sample in sample_list:
    adata = sc.read_h5ad(f"data/Ren_processed/{sample}.h5ad")
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    adatas_tfmrna[sample] = adata[:, [gene for gene in tf_list if gene in adata.var_names]]
    adatas_markers[sample] = adata[:, [gene for gene in marker_genes if gene in adata.var_names]]
    adatas_lrs[sample] = adata[:, [gene for gene in lr_genes if gene in adata.var_names]]

In [7]:
path = 'dataframes/morani_ren/'
if not os.path.exists(path):
    os.mkdir(path)

## Compute Moran's I

In [8]:
fname = f'{path}/mrna_marker_per_sample.csv'
if os.path.isfile(fname):
    moran_mran_df = pd.read_csv(fname, index_col=0)
else:
    moran_mran_df = stan.compute_morans_i(adatas_markers)
    moran_mran_df.to_csv(fname)

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

In [9]:
fname = f'{path}/mrna_lr_per_sample.csv'
if os.path.isfile(fname):
    moran_mran_df = pd.read_csv(fname, index_col=0)
else:
    moran_mran_df = stan.compute_morans_i(adatas_markers)
    moran_mran_df.to_csv(fname)

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

In [10]:
fname = f'{path}/mrna_tf_per_sample.csv'
if os.path.isfile(fname):
    moran_mran_df = pd.read_csv(fname, index_col=0)
else:
    moran_mran_df = stan.compute_morans_i(adatas_tfmrna)
    moran_mran_df.to_csv(fname)

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

In [11]:
fname = f'{path}/tfa_per_sample.csv'
if os.path.isfile(fname):
    moran_tfa_df = pd.read_csv(fname, index_col=0)
else:
    moran_tfa_df = stan.compute_morans_i(adatas_tfa)
    moran_tfa_df.to_csv(fname)

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

In [12]:
fname = f'{path}/pwa_per_sample.csv'
if os.path.isfile(fname):
    moran_pwa_df = pd.read_csv(fname, index_col=0)
else:
    moran_pwa_df = stan.compute_morans_i(adatas_pwa)
    moran_pwa_df.to_csv(fname)

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

## Compute bivariate Moran's I

In [13]:
fname = f'{path}/bivariate_tfa_tfa_per_sample.csv'
if os.path.isfile(fname):
    bivariate_tfa_tfa_df = pd.read_csv(fname, index_col=0)
else:
    bivariate_tfa_tfa_df = stan.compute_bivariate_morans_i(adatas_tfa, adatas_tfa, is_symmetric=True)
    bivariate_tfa_tfa_df.to_csv(fname)

In [14]:
fname = f'{path}/bivariate_tfa_marker_per_sample.csv'
if os.path.isfile(fname):
    bivariate_tfa_mrna_df = pd.read_csv(fname, index_col=0)
else:
    bivariate_tfa_mrna_df = stan.compute_bivariate_morans_i(adatas_tfa, adatas_markers)
    bivariate_tfa_mrna_df.to_csv(fname)

In [15]:
fname = f'{path}/bivariate_tfa_marker_per_sample.csv'
bivariate_tfa_mrna_df = stan.compute_bivariate_morans_i(adatas_tfa, adatas_markers)
bivariate_tfa_mrna_df.to_csv(fname)

In [16]:
fname = f'{path}/bivariate_tfa_lr_per_sample.csv'
if os.path.isfile(fname):
    bivariate_tfa_mrna_df = pd.read_csv(fname, index_col=0)
else:
    bivariate_tfa_mrna_df = stan.compute_bivariate_morans_i(adatas_tfa, adatas_lrs)
    bivariate_tfa_mrna_df.to_csv(fname)

In [17]:
fname = f'{path}/bivariate_tfa_pwa_per_sample.csv'
if os.path.isfile(fname):
    bivariate_tfa_pwa_df = pd.read_csv(fname, index_col=0)
else:
    bivariate_tfa_pwa_df = stan.compute_bivariate_morans_i(adatas_tfa, adatas_pwa)
    bivariate_tfa_pwa_df.to_csv(fname)